<figure>
  <IMG SRC="https://upload.wikimedia.org/wikipedia/commons/thumb/d/d5/Fachhochschule_Südwestfalen_20xx_logo.svg/320px-Fachhochschule_Südwestfalen_20xx_logo.svg.png" WIDTH=250 ALIGN="right">
</figure>

# Einführung Machine Learning
### Sommersemester 2026
Prof. Dr. Heiner Giefers

# Klassifikation mit Entscheidungsbäumen und Random Forest

In dieser Übung verwenden wir den bekannten Titanic-Datensatz, um zwei weitere Klassifikationsmodelle auszuprobieren:

- Entscheidungsbaum
- Random Forest (als Ensemble-Methode)

Die Vorverarbeitung führen wir hier nur kurz aus, da sie bereits im vorherigen Notebook ausführlich behandelt wurde.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Daten laden
url = "https://raw.githubusercontent.com/fhswf/datasets/refs/heads/main/Titanic_de.csv"
df = pd.read_csv(url)

# Aufteilen
y = df["ueberlebt"]
X = df.drop(columns=["ueberlebt"])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

## Vorverarbeitung

Wir schauen uns zuerst die fehlende Werte des Datensatzes an.
Auch bei Entscheidungsbäumen (in `scikit-learn`) müssen fehlende Werte (`NaN`) vor dem Training ersetzt (imputiert) werden.
Andernfalls kann dies zu einem Fehler beim Modelltraining oder bei der Vorhersage führen.

Die folgende Code-Zelle berechnet die Features mit fehlenden Werten und stellt sie, zusammen mit dem Datentyp des Features dar:

In [ ]:
fehlende = (X_train.isnull().mean()*100).to_dict()
# Lösche Elemente mit value 0
fehlende = {k: v for k, v in fehlende.items() if v != 0}
# Sortiert nach Wert (aufsteigend)
fehlende = dict(sorted(fehlende.items(), key=lambda item: item[1]))
# Gib Spalten mit fehlenden Werten aus:
for k, v in fehlende.items():
    print(f"Feature {k} mit {v:.2f} fehlenden Werten (Datentyp ist: {X_train[k].dtype})")

**Aufgabe:** Verwenden Sie den `SimpleImputer` aus `sklearn`, um die fehlende. Werte zu ersetzen.
Verwenden Sie den Mittelwert (`strategy="mean"`) für die numerischen Merkmale und den Modus (`strategy="most_frequent"`) für die kategorischen Merkmale.

In [ ]:
imputer_num = None
imputer_cat = None

# YOUR CODE HERE
raise NotImplementedError()

X_train[["alter"]] = imputer_num.transform(X_train[["alter"]])
X_train[["deck", "hafen"]] = imputer_cat.transform(X_train[["deck", "hafen"]])

X_test[["alter"]] = imputer_num.transform(X_test[["alter"]])
X_test[["deck", "hafen"]] = imputer_cat.transform(X_test[["deck", "hafen"]])

In [ ]:
# TESTS
assert not X_train[["alter", "deck", "hafen"]].isnull().any().any(), "Es gibt noch fehlende Werte in den Trainingsdaten."
assert not X_test[["alter", "deck", "hafen"]].isnull().any().any(), "Es gibt noch fehlende Werte in den Testdaten."

## Kodierung kategorialer Merkmale

Entscheidungsbäume erfordern numerische Eingaben, können aber gut mit kategorialen Merkmalen umgehen – solange diese zuvor in Zahlen umgewandelt wurden.

Da wir unseren Datensatz bereits in Trainings- und Testdaten (`X_train`, `X_test`) aufgeteilt haben, müssen wir sicherstellen, dass **beide Teilmengen die gleiche Kodierung erhalten**.

- Ordinale Merkmale wie `klasse` oder `geschlecht` können mit einer festen Zuordnung (`map`) in numerische Werte umgewandelt werden.
- Boolesche Merkmale wie `erwachsener_mann` oder `allein` werden einfach in `int` konvertiert.
- Nominale Merkmale wie `deck` oder `hafen` werden mit **Label-Encoding** (`.astype("category").cat.codes`) in Ganzzahlen übersetzt – das ist kompakt und für Entscheidungsbäume ausreichend.

Wir wenden die Kodierung auf beide DataFrames (`X_train`, `X_test`) gleichzeitig an, um konsistente Wertebereiche zu gewährleisten.

In [ ]:
# Manuelle Kodierung für ordinale Merkmale
map_geschlecht = {"weiblich": 0, "maennlich": 1}
map_klasse = {"Erste": 0, "Zweite": 1, "Dritte": 2}

# Wende die Kodierungen auf X_train und X_test gleich an
for df in [X_train, X_test]:
    df["geschlecht"] = df["geschlecht"].map(map_geschlecht)
    df["klasse"] = df["klasse"].map(map_klasse)
    df[["erwachsener_mann", "allein"]] = df[["erwachsener_mann", "allein"]].astype(int)
    
    # Label-Encoding für nominale Features
    for col in ["deck", "hafen"]:
        df[col] = df[col].astype("category").cat.codes

## Hinweis zur Vorverarbeitung bei Entscheidungsbäumen

Entscheidungsbäume sind robuste Modelle, die auf **Schwellenvergleichen** basieren (z. B. `alter < 15`). Daher spielt der **absolute Wertebereich** der Merkmale keine Rolle.

Im Gegensatz zu vielen anderen Modellen (z. B. KNN, logistische Regression, SVM) ist bei Entscheidungsbäumen **keine Normalisierung oder Standardisierung notwendig**:

- Ein Merkmal mit Werten im Bereich `0–1` wird **gleich behandelt** wie eines im Bereich `0–1000`.
- Auch sehr unterschiedlich skalierte Merkmale beeinflussen die Entscheidungsstruktur **nicht negativ**.

Bei Entscheidungsbäumen können wir also auf alle Schritte zur Skalierung der Daten verzichten – das spart Aufwand und vereinfacht die Vorverarbeitung deutlich.

## Entscheidungsbaum trainieren

**Aufgabe:** Trainieren Sie ein einfaches Klassifikationsmodell mit einem **Entscheidungsbaum** (`DecisionTreeClassifier`) auf den Trainingsdaten

Wir geben eine maximale Tiefe des Baumes (`max_depth=3`) vor, um **Overfitting** zu vermeiden und den Baum **übersichtlich** zu halten.  
Das trainierte Modell kann anschließend zur Klassifikation neuer Fälle verwendet werden.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# YOUR CODE HERE
raise NotImplementedError()

In [ ]:
### Test
assert isinstance(tree, DecisionTreeClassifier), "Das Modell ist kein DecisionTreeClassifier."
assert tree.get_depth() <= 4, "Der Baum ist tiefer als gewünscht (max_depth=4)."
assert hasattr(tree, "feature_importances_"), "Das Modell wurde noch nicht trainiert."

## Bewertung des Entscheidungsbaum-Modells

Nach dem Training des Entscheidungsbaums möchten wir nun überprüfen, wie gut das Modell auf den **Testdaten** abschneidet.

**Aufgabe:** Berechnen Sie die Vorhersagen (`y_pred_tree`) des Entscheidungsbaum-Modells für die Testdaten sowie die Accuracy (`acc_tree`) der Vorhersage.

In [ ]:
from sklearn.metrics import classification_report, accuracy_score

y_pred_tree = None
acc_tree = None

# YOUR CODE HERE
raise NotImplementedError()

print("Entscheidungsbaum – Accuracy: %.2f%%" % (acc_tree*100))
print(classification_report(y_test, y_pred_tree))

In [ ]:
### Test
assert y_pred_tree is not None, "Die Vorhersage fehlt."
assert acc_tree is not None, "Accuracy wurde nicht berechnet."
assert isinstance(acc_tree, float), "Accuracy sollte ein Float-Wert sein."
assert 0.0 <= acc_tree <= 1.0, "Accuracy liegt außerhalb des gültigen Bereichs (0–1)."

## Visualisierung des Entscheidungsbaums

Ein großer Vorteil von Entscheidungsbäumen ist ihre **gute Interpretierbarkeit**.  
Wir können den gelernten Baum als Grafik darstellen und so nachvollziehen, wie das Modell Entscheidungen trifft.

In der Visualisierung sehen wir:
- die verwendeten Merkmale in jedem Knoten,
- die aufgeteilten Daten (Samples pro Klasse),
- und die Klassenvorhersage mit Farbcodierung.

So lässt sich leicht ablesen, welche Bedingungen zur Entscheidung „überlebt“ oder „gestorben“ führen.

In [ ]:
from sklearn.tree import plot_tree
import matplotlib.pyplot as plt

plt.figure(figsize=(20, 8))
plot_tree(
    tree,
    feature_names=X_train.columns,
    class_names=["gestorben", "überlebt"],
    filled=True,
    rounded=True,
    fontsize=10
)
plt.title("Entscheidungsbaum für Titanic-Daten")
plt.show()

## Random Forest trainieren

Ein Random Forest besteht aus vielen Entscheidungsbäumen, die auf unterschiedlichen Teilmengen der Daten trainiert werden. Durch diese **Ensemble-Methode (Bagging)** lassen sich stabilere und genauere Vorhersagen erzielen als mit einem einzelnen Entscheidungsbaum.

**Aufgabe:** Trainieren Sie ein Random-Forest-Modell mit `scikit-learn`. Setzen Sie die Parameter wie folgt:
- `n_estimators=50`: Anzahl der Entscheidungsbäume
- `max_depth=4`: maximale Tiefe der einzelnen Bäume
- `random_state=42`: für Reproduzierbarkeit

In [ ]:
from sklearn.ensemble import RandomForestClassifier
forest = None

# YOUR CODE HERE
raise NotImplementedError()

In [ ]:
### Testzelle: Random-Forest-Modell korrekt trainiert
assert forest is not None, "Das Random-Forest-Modell wurde nicht erstellt."
assert isinstance(forest, RandomForestClassifier), "Objekt ist kein RandomForestClassifier."
assert hasattr(forest, "predict"), "Das Modell wurde vermutlich nicht korrekt trainiert."

## Bewertung des Random-Forest-Modells

**Aufgabe:** Berechnen Sie die Vorhersagen (`y_pred_forest`) des Random-Forest-Modells für die Testdaten sowie die Accuracy (`acc_tree`) der Vorhersage.


In [ ]:
y_pred_forest = None
acc_forest = None

# YOUR CODE HERE
raise NotImplementedError()


print("Random Forest – Accuracy: %.2f%%" % (acc_forest*100))
print(classification_report(y_test, y_pred_forest))

In [ ]:
### Testzelle: Bewertung des Random-Forest-Modells
assert y_pred_forest is not None, "Die Vorhersage fehlt."
assert isinstance(y_pred_forest, (list, np.ndarray)), "Vorhersage hat falschen Typ."
assert accuracy_score(y_test, y_pred_forest) > 0.6, "Accuracy liegt unter 60% – möglicherweise wurde das Modell nicht korrekt trainiert."

## Vergleich: Decision Tree vs. Random Forest

Obwohl Random Forests in vielen Anwendungen eine **deutlich bessere Leistung** als einzelne Entscheidungsbäume zeigen, kann es in bestimmten Fällen vorkommen, dass der **Decision Tree besser abschneidet** – insbesondere wenn:

- die optimale Modellkomplexität sehr niedrig ist (z. B. flacher Baum),
- das Ensemble durch `max_depth` stark eingeschränkt wird,
- oder die zufällige Stichprobenbildung bei `RandomForest` nicht zur Struktur der Daten passt.

Ein einzelner Entscheidungsbaum kann sich in solchen Fällen stärker an die Trainingsdaten anpassen und dadurch auf kleinen, gut strukturierten Datensätzen höhere Genauigkeit erzielen.

Um ein Modell zu erhalten, das Fehler systematisch reduziert, probieren wir als Nächstes einen **Boosting-Ansatz mit dem `XGBClassifier`** aus. Boosting-Methoden wie XGBoost bauen sequentiell mehrere schwache Modelle auf, wobei jedes Modell gezielt die Fehler der vorherigen ausgleicht.

Auch Boosting-Modelle wie XGBoost garantieren keine besseren Ergebnisse – sie sind leistungsfähig, aber empfindlicher gegenüber Rauschen und erfordern sorgfältige Parametrierung.

In [ ]:
try:
    from xgboost import XGBClassifier
except:
    import sys
    !{sys.executable} -m pip install xgboost
    from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    eval_metric="logloss",
    random_state=42
)

xgb.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import accuracy_score

y_pred_xgb = xgb.predict(X_test)
acc_xgb = accuracy_score(y_test, y_pred_xgb)
print("XGBoost – Accuracy: %.2f%%" % (acc_xgb * 100))

In [ ]:
y_pred_xgb = xgb.predict(X_test)
acc_xgb = accuracy_score(y_test, y_pred_xgb)

print("XGBoost – Accuracy: %.2f%%" % (acc_xgb*100))
print(classification_report(y_test, y_pred_xgb))

In [ ]:
print("Decision Tree – Accuracy: %.2f%%" % (acc_tree*100))
print("Random Forest – Accuracy: %.2f%%" % (acc_forest*100))
print("XGBoost       – Accuracy: %.2f%%" % (acc_xgb*100))